# Behavior Cloning Data Collection

Pipeline: HuggingFace replays → padded maps + env actions → `lax.scan` one game → `vmap` a batch → loop over **all** games → `(game_id, obs, action)` pairs.

Architecture (from `sample_game_runner.py`):
1. **Convert** replay maps / moves into env grids + `(T, 2, 5)` action sequences
2. **`scan_one_game`** — `jax.lax.scan` over timesteps; collect `obs.as_tensor()` then `step`
3. **`run_batch`** — `jax.vmap(scan_one_game)` over N=32/64 games
4. **Loop** over the full dataset in batches of `BATCH_SIZE`, writing one `.npz` shard per batch


## 1. Imports & config

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import Any

# Resolve repo root so `import generals` works from any notebook cwd.
def _find_repo_root() -> Path:
    candidates: list[Path] = []
    # Cursor / VS Code Jupyter exposes the notebook path
    nb_file = globals().get("__vsc_ipynb_file__")
    if nb_file:
        candidates.append(Path(nb_file).resolve().parent)
    candidates.append(Path.cwd().resolve())
    candidates.append(Path.cwd().resolve() / "bc")
    candidates.append(Path.cwd().resolve().parent)

    seen: set[Path] = set()
    for start in candidates:
        for cand in [start, *start.parents]:
            if cand in seen:
                continue
            seen.add(cand)
            if (cand / "generals" / "__init__.py").is_file():
                return cand
    raise ImportError(
        "Could not locate generals/ package. "
        f"Tried from cwd={Path.cwd()} — open the repo root or bc/ as the notebook folder."
    )


_ROOT = _find_repo_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import jax
import jax.numpy as jnp
import numpy as np
from datasets import load_dataset

from generals import create_action, create_initial_state, step, get_observation
from generals.core.game import GameInfo, GameState

print(f"generals import ok  (root={_ROOT})")

# Fixed shapes for vmap (maps in this dataset are 17–23; turns go up to 1000)
PAD_TO = 23
TRUNCATION = 1001         # demo length; raise toward 1001 for full games
BATCH_SIZE = 32           # try 64 once the compile fits in memory
SEED = 0
INCLUDE_PASSES = False    # if False, BC pairs keep only real moves (pass==0)
MAX_GAMES = 500         # None = all replays; set e.g. 128 to smoke-test a few batches


## 2. Load HuggingFace replays

In [16]:
print("Loading dataset...")
dataset = load_dataset("strakammm/generals_io_replays")
train_dataset = dataset["train"]
print(f"Replays: {len(train_dataset)}")
print(f"Columns: {train_dataset.column_names}")


Loading dataset...
Replays: 18803
Columns: ['version', 'id', 'mapWidth', 'mapHeight', 'usernames', 'stars', 'cities', 'cityArmies', 'generals', 'mountains', 'moves']


## 3. Action conversion

Dataset move: `[player_index, start_tile, end_tile, is_50%_move, turn_number]`

Env action: `[pass, row, col, direction, split]` with direction `0↑ 1↓ 2← 3→`.

Tile index → `(row, col)` via `row, col = divmod(tile, mapWidth)`.


In [17]:
PASS = np.asarray(create_action(to_pass=True), dtype=np.int32)

DELTA_TO_DIR = {
    (-1, 0): 0,  # UP
    (1, 0): 1,   # DOWN
    (0, -1): 2,  # LEFT
    (0, 1): 3,   # RIGHT
}


def tile_to_rc(tile: int, width: int) -> tuple[int, int]:
    return divmod(int(tile), int(width))


def tiles_to_direction(start_tile: int, end_tile: int, width: int) -> int:
    sr, sc = tile_to_rc(start_tile, width)
    er, ec = tile_to_rc(end_tile, width)
    key = (er - sr, ec - sc)
    if key not in DELTA_TO_DIR:
        raise ValueError(f"non-adjacent move {start_tile}->{end_tile} (delta={key})")
    return DELTA_TO_DIR[key]


def dataset_move_to_action(start_tile: int, end_tile: int, is_50: int, width: int) -> np.ndarray:
    """Convert one dataset move into env action [pass, row, col, direction, split]."""
    row, col = tile_to_rc(start_tile, width)
    direction = tiles_to_direction(start_tile, end_tile, width)
    return np.array([0, row, col, direction, int(is_50)], dtype=np.int32)


# quick sanity check
_w = 19
_a = dataset_move_to_action(20, 39, 0, _w)  # down one row
assert _a.tolist() == [0, 1, 1, 1, 0], _a
print("action conversion ok:", _a)


action conversion ok: [0 1 1 1 0]


## 4. Map + action-sequence builders

Grid encoding for `create_initial_state`:
`-2` mountain · `0` empty · `1` P0 general · `2` P1 general · `>2` city army value

Pads every map to `(PAD_TO, PAD_TO)` with mountains so a batch can be `vmap`'d.


In [18]:
def replay_to_grid(replay: dict[str, Any], pad_to: int = PAD_TO) -> np.ndarray:
    """Build a padded numeric grid from one HF replay."""
    h, w = int(replay["mapHeight"]), int(replay["mapWidth"])
    if h > pad_to or w > pad_to:
        raise ValueError(f"map {(h, w)} exceeds pad_to={pad_to}")

    grid = np.zeros((h, w), dtype=np.int32)

    for tile in replay["mountains"]:
        r, c = tile_to_rc(tile, w)
        grid[r, c] = -2

    for tile, army in zip(replay["cities"], replay["cityArmies"]):
        r, c = tile_to_rc(tile, w)
        grid[r, c] = int(army)

    for player, tile in enumerate(replay["generals"]):
        r, c = tile_to_rc(tile, w)
        grid[r, c] = player + 1  # 1 or 2

    padded = np.full((pad_to, pad_to), -2, dtype=np.int32)
    padded[:h, :w] = grid
    return padded


def replay_to_actions(replay: dict[str, Any], truncation: int = TRUNCATION) -> np.ndarray:
    """
    Build a (T, 2, 5) action sequence.

    Turns with no recorded move for a player stay as pass.
    Moves with turn_number >= truncation are dropped.
    """
    w = int(replay["mapWidth"])
    seq = np.broadcast_to(np.stack([PASS, PASS]), (truncation, 2, 5)).copy()

    for move in replay["moves"]:
        player, start, end, is_50, turn = (int(x) for x in move)
        if turn >= truncation:
            continue
        seq[turn, player] = dataset_move_to_action(start, end, is_50, w)

    return seq


# smoke test on one replay
_sample = train_dataset[0]
_grid = replay_to_grid(_sample)
_acts = replay_to_actions(_sample, truncation=64)
print(f"grid {_grid.shape}, nonzero={np.count_nonzero(_grid != -2)}")
print(f"actions {_acts.shape}, non-pass moves={int(np.sum(_acts[:, :, 0] == 0))}")


grid (23, 23), nonzero=295
actions (64, 2, 5), non-pass moves=81


## 5. Batch builders — stack N replays into vmappable arrays

In [19]:
def build_batch(
    replays: list[dict[str, Any]],
    truncation: int = TRUNCATION,
    pad_to: int = PAD_TO,
) -> tuple[GameState, jnp.ndarray]:
    """
    Convert a list of replays into batched initial states + actions.

    Returns:
        initial_states: GameState pytree with leading axis N
        actions: (N, T, 2, 5) int32
    """
    grids = jnp.stack([jnp.asarray(replay_to_grid(r, pad_to)) for r in replays])
    actions = jnp.stack([jnp.asarray(replay_to_actions(r, truncation)) for r in replays])
    initial_states = jax.vmap(create_initial_state)(grids)
    return initial_states, actions


def sample_replays(dataset, n: int, seed: int = SEED) -> list[dict[str, Any]]:
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(dataset), size=n, replace=False)
    return [dataset[int(i)] for i in idxs]


print("build_batch ready")


build_batch ready


## 6. `scan_one_game` — one full rollout with `lax.scan`

At each timestep:
1. Build fogged `tensor_obs` for both players via `get_observation(...).as_tensor()` → `(2, 14, H, W)`
2. Apply the joint action with `step`
3. Carry the next state; emit `(obs, action, info)`


In [20]:
def scan_one_game(
    initial_state: GameState,
    actions: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray, GameInfo]:
    """
    Run one game for T steps.

    Args:
        initial_state: single GameState
        actions: (T, 2, 5)

    Returns:
        obs:   (T, 2, 14, H, W)  — tensor obs *before* each action
        acts:  (T, 2, 5)         — same as input (echoed for convenience)
        infos: GameInfo with leading time axis (T, ...)
    """

    def body(state: GameState, action: jnp.ndarray):
        obs0 = get_observation(state, 0).as_tensor()
        obs1 = get_observation(state, 1).as_tensor()
        obs = jnp.stack([obs0, obs1], axis=0)  # (2, 14, H, W)
        new_state, info = step(state, action)
        return new_state, (obs, action, info)

    _final, (obs, acts, infos) = jax.lax.scan(body, initial_state, actions)
    return obs, acts, infos


print("scan_one_game defined")


scan_one_game defined


## 7. `run_batch` — `vmap` over N games (then `jit`)

Shapes in / out:
- `initial_states`: batched `GameState` `(N, ...)`
- `actions`: `(N, T, 2, 5)`
- `obs`: `(N, T, 2, 14, H, W)`
- `acts`: `(N, T, 2, 5)`
- `infos`: `(N, T, ...)`


In [21]:
@jax.jit
def run_batch(
    initial_states: GameState,
    actions: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray, GameInfo]:
    """Parallel full-game rollouts via vmap(scan_one_game)."""
    return jax.vmap(scan_one_game)(initial_states, actions)


print("run_batch defined (jit + vmap)")


run_batch defined (jit + vmap)


## 8. Flatten trajectories → supervised `(obs, action)` + metadata

Each player-timestep is one BC example: `game_id` · `obs` → `action`.

Also stores `player` and `turn`. By default we drop passes (`pass==1`).


In [22]:
def trajectories_to_bc_pairs(
    obs: jnp.ndarray,
    acts: jnp.ndarray,
    game_ids: list[str] | np.ndarray,
    include_passes: bool = INCLUDE_PASSES,
) -> dict[str, np.ndarray]:
    """
    Flatten batched trajectories into supervised pairs + per-example metadata.

    Args:
        obs:      (N, T, 2, 14, H, W)
        acts:     (N, T, 2, 5)
        game_ids: length-N list/array of replay ids (one per game in the batch)

    Returns dict:
        obs:      (M, 14, H, W)
        actions:  (M, 5)
        game_id:  (M,) unicode — replay id for each pair
        player:   (M,) int32  — 0 or 1
        turn:     (M,) int32  — timestep index within the game
    """
    n, t, p = acts.shape[:3]
    assert len(game_ids) == n, f"expected {n} game_ids, got {len(game_ids)}"

    x = np.asarray(obs).reshape(n * t * p, *obs.shape[3:])
    y = np.asarray(acts).reshape(n * t * p, 5)

    # metadata aligned with the same (n, t, p) flatten order
    game_id = np.repeat(np.asarray(game_ids, dtype=object), t * p)
    player = np.tile(np.arange(p, dtype=np.int32), n * t)
    turn = np.repeat(np.arange(t, dtype=np.int32), p)
    turn = np.tile(turn, n)

    if not include_passes:
        mask = y[:, 0] == 0  # real moves only
        x, y = x[mask], y[mask]
        game_id, player, turn = game_id[mask], player[mask], turn[mask]

    return {
        "obs": x,
        "actions": y,
        "game_id": game_id.astype(str),
        "player": player,
        "turn": turn,
    }


print("trajectories_to_bc_pairs defined")


trajectories_to_bc_pairs defined


## 9. Warmup compile (one batch)

JIT-compile `run_batch` on a single batch before the full sweep.


In [23]:
# Warmup: one batch so the first full-loop iteration is not paying compile cost.
_warmup = [train_dataset[i] for i in range(min(BATCH_SIZE, len(train_dataset)))]
_ws, _wa = build_batch(_warmup, truncation=TRUNCATION, pad_to=PAD_TO)
_ = run_batch(_ws, _wa)
print(f"warmup done  batch={len(_warmup)}  T={TRUNCATION}  pad={PAD_TO}")


warmup done  batch=32  T=1001  pad=23


## 10. Single-game sanity check (optional)

Replay game 0 through `scan_one_game` and confirm every non-pass action was legal
under fogged ownership at decision time.


In [24]:
from generals.core.action import compute_valid_move_mask

def count_illegal_moves(replay: dict[str, Any], truncation: int = TRUNCATION) -> dict[str, int]:
    grid = jnp.asarray(replay_to_grid(replay))
    actions = jnp.asarray(replay_to_actions(replay, truncation))
    state = create_initial_state(grid)

    illegal = 0
    applied = 0
    for t in range(truncation):
        action = actions[t]
        for p in range(2):
            a = action[p]
            if int(a[0]) == 1:
                continue
            applied += 1
            o = get_observation(state, int(p))
            mask = compute_valid_move_mask(o.armies, o.owned_cells, o.mountains)
            if not bool(mask[int(a[1]), int(a[2]), int(a[3])]):
                illegal += 1
        state, info = step(state, action)
        if bool(info.is_done):
            # remaining actions are pads / post-game
            break

    return {
        "applied": applied,
        "illegal": illegal,
        "winner": int(info.winner),
        "end_time": int(info.time),
    }


stats = count_illegal_moves(train_dataset[0], truncation=TRUNCATION)
print(stats)
assert stats["illegal"] == 0, stats


{'applied': 428, 'illegal': 0, 'winner': 0, 'end_time': 241}


## 11. Collect **all** games — batched `vmap` loop

Walks the full HuggingFace split in chunks of `BATCH_SIZE`:

1. `build_batch` → grids + `(N, T, 2, 5)` actions  
2. `run_batch` (`jit` + `vmap(scan_one_game)`)  
3. flatten to BC pairs  
4. write `bc/data/bc_shard_{shard:04d}.npz`

The last chunk is padded to `BATCH_SIZE` for a stable JIT shape; padded slots are dropped before saving.

Set `MAX_GAMES` in config to limit how many replays to process (e.g. `128` for a quick test).


In [ ]:
from pathlib import Path

OUT_DIR = Path("data_noaug")
OUT_DIR.mkdir(parents=True, exist_ok=True)

n_total = len(train_dataset) if MAX_GAMES is None else min(int(MAX_GAMES), len(train_dataset))
n_shards = (n_total + BATCH_SIZE - 1) // BATCH_SIZE
print(f"collecting {n_total}/{len(train_dataset)} games  |  batch={BATCH_SIZE}  shards={n_shards}  T={TRUNCATION}")
print(f"output dir: {OUT_DIR.resolve()}")

shard_paths: list[Path] = []
total_pairs = 0
total_games = 0

for shard_idx, start in enumerate(range(0, n_total, BATCH_SIZE)):
    end = min(start + BATCH_SIZE, n_total)
    n_real = end - start

    replays = [train_dataset[i] for i in range(start, end)]
    # Pad to BATCH_SIZE so JIT shape stays fixed across shards.
    if n_real < BATCH_SIZE:
        replays = replays + [replays[0]] * (BATCH_SIZE - n_real)

    game_ids = [str(r["id"]) for r in replays]
    initial_states, actions = build_batch(replays, truncation=TRUNCATION, pad_to=PAD_TO)
    obs, acts, infos = run_batch(initial_states, actions)

    # Drop padded slots before flattening.
    obs = obs[:n_real]
    acts = acts[:n_real]
    game_ids = game_ids[:n_real]

    bc = trajectories_to_bc_pairs(obs, acts, game_ids)

    out_path = OUT_DIR / f"bc_shard_{shard_idx:04d}_n{n_real}_t{TRUNCATION}.npz"
    tmp_path = out_path.with_suffix(".npz.tmp")
    np.savez_compressed(
        tmp_path,
        obs=bc["obs"],
        actions=bc["actions"],
        game_id=bc["game_id"],
        player=bc["player"],
        turn=bc["turn"],
    )
    tmp_path.replace(out_path)  # atomic finalize — avoids half-written shards
    shard_paths.append(out_path)
    total_pairs += int(bc["obs"].shape[0])
    total_games += n_real

    winners = infos.winner[:n_real, -1]
    print(
        f"shard {shard_idx+1:4d}/{n_shards}  games[{start}:{end}]  "
        f"pairs={bc['obs'].shape[0]:,}  "
        f"P0={int(jnp.sum(winners == 0))} P1={int(jnp.sum(winners == 1))} "
        f"unfin={int(jnp.sum(winners < 0))}  → {out_path.name}"
    )

print(f"\ndone. games={total_games:,}  pairs={total_pairs:,}  shards={len(shard_paths)}")
print(f"first: {shard_paths[0]}")
print(f"last:  {shard_paths[-1]}")
